# LLM Request SDK Quickstart

서버화된 `llm-request`를 SDK로 사용하는 가이드.

**구조:**
```
이 노트북 (SDK)                     llm-request Service (port 8101)
┌──────────────────┐                ┌─────────────────────────┐
│ checksum 계산     │   HTTP        │ MDB 캐시 조회 (LLM 응답) │
│ Redis 파일캐시    │ ───────────→  │ 파일 처리 (Files API)    │
│ 서버 호출         │               │ LLM 호출 (Gemini)       │
│ 결과 수신         │ ←─────────── │ 결과 저장 + 반환         │
└──────────────────┘                └─────────────────────────┘
```

**기존 라이브러리와 차이점:**
- 설정: handler URL 3개 → `LLM_REQUEST_URL` 1개
- 사용 코드: 동일 (`llm_request_async(...)`)
- 내부 동작: 직접 실행 → HTTP로 서버에 위임

## 설정

In [10]:
import os

# SDK는 서비스 URL 하나만 설정하면 됨
# os.environ["LLM_REQUEST_URL"] = "http://192.7.9.45:8101"
os.environ["LLM_REQUEST_URL"] = "http://host.docker.internal:8101"


# Redis (SDK에서 파일 캐시 조회용)
os.environ["REDIS_URL"] = "redis://192.7.9.45:7617/0"

from llm_request import configure
configure()

## 타이밍 유틸리티

각 단계별 소요 시간과 네트워크 흐름을 추적하는 헬퍼.

In [11]:
import time
from contextlib import contextmanager

@contextmanager
def timer(label: str):
    """구간별 소요 시간 측정."""
    start = time.perf_counter()
    print(f"  [{label}] 시작...")
    yield
    elapsed = (time.perf_counter() - start) * 1000
    print(f"  [{label}] 완료: {elapsed:.0f}ms")

def print_flow(title: str, meta: dict, elapsed_ms: float):
    """요청 흐름 요약 출력."""
    source = meta.get("source", "?")
    cache_hit = meta.get("cache_hit", False)
    cache_key = meta.get("cache_key", "")[:16]
    
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    print(f"  소스:      {source} ({'MDB 캐시 히트' if cache_hit else 'LLM 호출'})")
    print(f"  캐시 키:   {cache_key}...")
    print(f"  총 소요:   {elapsed_ms:.0f}ms")
    
    if meta.get("route_reason"):
        print(f"  경로:      {meta['route_reason']}")
    print()

---
## 1. 기본 호출 — 텍스트만 (파일 없음)

```
SDK                              Service (8101)
 │  POST /v1/request             │
 │  {llm_input, cache_mode}  ──→ │ → MDB 캐시 조회
 │                               │ → MISS → LLM 호출 (Gemini)
 │  ←── {result, meta}          │ → MDB 캐시 저장
```

In [12]:
from llm_request import llm_request_async

# --- 1회차: LLM 직접 호출 (캐시 miss) ---
start = time.perf_counter()
result, meta = await llm_request_async(
    {
        "contents": "1+1은?",
        "model": "gemini-2.5-flash",
    },
    database="kw_ai_dev",
)
elapsed = (time.perf_counter() - start) * 1000

print_flow("1회차: 첫 호출 (캐시 miss → LLM)", meta, elapsed)
print(f"  응답: {result.get('text', '')[:100]}")


  1회차: 첫 호출 (캐시 miss → LLM)
  소스:      cache (MDB 캐시 히트)
  캐시 키:   81c7c9133a10d5e9...
  총 소요:   27ms

  응답: 1+1은 2입니다.


In [13]:
# --- 2회차: 동일 입력 → MDB 캐시 히트 (LLM 호출 없음) ---
start = time.perf_counter()
result2, meta2 = await llm_request_async(
    {
        "contents": "1+1은?",
        "model": "gemini-2.5-flash",
    },
    database="kw_ai_dev",
)
elapsed2 = (time.perf_counter() - start) * 1000

print_flow("2회차: 동일 입력 (MDB 캐시 히트)", meta2, elapsed2)
print(f"  응답: {result2.get('text', '')[:100]}")
print(f"  속도 비교: {elapsed:.0f}ms → {elapsed2:.0f}ms ({elapsed/max(elapsed2,0.1):.1f}x 빠름)")


  2회차: 동일 입력 (MDB 캐시 히트)
  소스:      cache (MDB 캐시 히트)
  캐시 키:   81c7c9133a10d5e9...
  총 소요:   15ms

  응답: 1+1은 2입니다.
  속도 비교: 27ms → 15ms (1.9x 빠름)


---
## 2. 파일 포함 호출

```
SDK (로컬)                              Service (8101)
 │                                       │
 │  1. checksum 계산 (로컬 SHA-256)       │
 │  2. Redis 파일 캐시 조회 (로컬)         │
 │     miss → POST /v1/files             │
 │     (파일 바이너리 전송)           ──→  │ → Files API 업로드
 │                                       │ → Redis 파일 캐시 저장
 │  3. POST /v1/request                  │
 │     (checksum만 전송)             ──→  │ → MDB 캐시 조회
 │                                       │ → MISS → LLM 호출
 │  ←── {result, meta}                  │ → MDB 캐시 저장
```

In [16]:
from pathlib import Path
from llm_request import llm_request_async
from llm_request.checksum import file_checksum

# 테스트용 파일 (quickstart.pdf 또는 아무 PDF)
test_file = "quickstart.pdf"
if not Path(test_file).exists():
    print(f"⚠️ {test_file}이 없습니다. 아래 셀은 건너뛰세요.")
else:
    # --- 단계별 타이밍 측정 ---
    print(f"파일: {test_file} ({Path(test_file).stat().st_size / 1024:.0f}KB)")
    print()

    # 1. checksum 계산 (로컬)
    with timer("checksum 계산 (로컬)"):
        cs = file_checksum(test_file)
    print(f"    → {cs['checksum'][:24]}... ({cs['mime_type']})")
    print()

    # 2. 전체 호출 (SDK가 내부에서 Redis 조회 + 파일 업로드 + 서버 요청)
    with timer("llm_request_async 전체"):
        result, meta = await llm_request_async(
            {
                "contents": "이 문서를 한 문장으로 요약해줘",
                "model": "gemini-2.5-flash",
            },
            file_paths=[test_file],
            database="kw_ai_dev",
        )
    
    print()
    print_flow("파일 포함 호출 (1회차)", meta, 0)
    print(f"  응답: {result.get('text', '')[:200]}")

파일: quickstart.pdf (598KB)

  [checksum 계산 (로컬)] 시작...
  [checksum 계산 (로컬)] 완료: 2ms
    → sha256:e6fdb7ac05ee86a25... (application/pdf)

  [llm_request_async 전체] 시작...
  [llm_request_async 전체] 완료: 25ms


  파일 포함 호출 (1회차)
  소스:      cache (MDB 캐시 히트)
  캐시 키:   723c717a4b0e4928...
  총 소요:   0ms

  응답: 이 문서는 KB금융그룹의 2024년 지속가능경영보고서로, 투자자 관점에서 기후위기 대응, 리스크 관리, 금융소비자보호, 디지털 혁신 및 기술, 포용금융 등 주요 지속가능성 이슈에 대한 그룹의 거버넌스, 전략, 위험관리, 지표 및 목표를 상세히 설명하고 있습니다.


---
## 3. done_check + fetch_by_cache_key

캐시 존재 여부만 확인하고, 필요 시 직접 조회.

```
SDK                              Service (8101)
 │  POST /v1/done-check          │
 │  {llm_input}             ──→  │ → MDB 캐시 조회
 │  ←── {cache_key, cached}     │
 │                               │
 │  POST /v1/fetch               │
 │  {cache_key}             ──→  │ → MDB에서 결과 반환
 │  ←── {result, meta}          │
```

In [17]:
from llm_request import done_check, fetch_by_cache_key

# --- done_check: 캐시 존재 여부 확인 ---
with timer("done_check"):
    dc = await done_check(
        {"contents": "1+1은?", "model": "gemini-2.5-flash"},
        database="kw_ai_dev",
    )

print(f"  cache_key: {dc['cache_key'][:16]}...")
print(f"  cache_hit: {dc['cache_hit']}")
print()

# --- fetch_by_cache_key: 캐시 키로 직접 조회 ---
if dc["cache_hit"]:
    with timer("fetch_by_cache_key"):
        result, meta = await fetch_by_cache_key(
            dc["cache_key"],
            database="kw_ai_dev",
        )
    print(f"  응답: {result.get('text', '')[:100]}")
else:
    print("  캐시 없음 — 위의 1번 셀을 먼저 실행하세요")

  [done_check] 시작...
  [done_check] 완료: 14ms
  cache_key: 81c7c9133a10d5e9...
  cache_hit: True

  [fetch_by_cache_key] 시작...
  [fetch_by_cache_key] 완료: 11ms
  응답: 1+1은 2입니다.


---
## 4. 기존 라이브러리와 비교

| | 라이브러리 (v1) | SDK (v2, 서버화) |
|--|---------------|----------------|
| **설정** | MDB_HANDLER_URL + LLM_HANDLER_URL + REDIS_URL + configure 3개 | LLM_REQUEST_URL 1개 |
| **의존성** | llm-handler, mdb-handler, redis-handler 전부 | httpx, redis-handler만 |
| **사용 코드** | `llm_request_async(...)` | `llm_request_async(...)` (동일) |
| **파일 처리** | 로컬에서 직접 Files API 호출 | checksum 계산(로컬) → 서버가 처리 |
| **캐시 조회** | 로컬에서 직접 MDB 조회 | 서버가 조회 |

---
## 5. 시스템 프롬프트 + service_name

In [ ]:
start = time.perf_counter()
result, meta = await llm_request_async(
    {
        "contents": "ESG 평가 기준을 알려줘",
        "model": "gemini-2.5-flash",
        "system": "너는 ESG 전문가야. 한국어로 답변해.",
        "temperature": 0,
        "max_tokens": 200,
        "service_name": "esg_agent",
    },
    database="kw_ai_dev",
)
elapsed = (time.perf_counter() - start) * 1000

print_flow("시스템 프롬프트 + service_name", meta, elapsed)
print(f"  응답: {result.get('text', '')[:200]}")

---
## 6. 대용량 파일 (>50MB PDF → 자동 split + Files API)

```
SDK (로컬)                                   Service (8101)
 │                                            │
 │  1. checksum 계산 (로컬)                    │
 │  2. >20MB → Redis 조회 → miss              │
 │  3. POST /v1/files (파일 전송)         ──→  │ → split_pdf (50MB 단위 분할)
 │                                            │ → 각 파트 Files API 업로드
 │                                            │ → Redis 파일 캐시 저장
 │  4. POST /v1/request (checksum만)     ──→  │ → MDB 캐시 조회 → MISS
 │                                            │ → Redis에서 URI 조회
 │                                            │ → LLM 호출
 │  ←── {result, meta}                       │ → MDB 캐시 저장
```

In [ ]:
from pathlib import Path

# 대용량 PDF (>50MB) — 경로를 환경에 맞게 수정
LARGE_PDF = str(Path("../../../Kiwoom-ESG/pdf_files/2024_아모레퍼시픽.pdf").resolve())

if not Path(LARGE_PDF).exists():
    print(f"⚠️ 대용량 PDF를 찾을 수 없습니다: {LARGE_PDF}")
    print("   경로를 수정하거나 이 셀을 건너뛰세요.")
else:
    size_mb = Path(LARGE_PDF).stat().st_size / (1024 * 1024)
    print(f"파일: {Path(LARGE_PDF).name} ({size_mb:.1f}MB)")
    print()

    # 6-1. 첫 호출 (split + upload + LLM)
    start = time.perf_counter()
    with timer("대용량 PDF 전체 (split + upload + LLM)"):
        result, meta = await llm_request_async(
            {
                "contents": "이 PDF의 제목을 한글 한 줄로.",
                "model": "gemini-2.5-flash",
                "max_tokens": 100,
            },
            file_paths=[LARGE_PDF],
            database="kw_ai_dev",
        )
    elapsed = (time.perf_counter() - start) * 1000

    print_flow("대용량 PDF (1회차)", meta, elapsed)
    print(f"  응답: {result.get('text', '')[:200]}")

In [ ]:
# 6-2. 같은 파일 재호출 → MDB 캐시 히트 (업로드/LLM 호출 없음)
if Path(LARGE_PDF).exists():
    start = time.perf_counter()
    result2, meta2 = await llm_request_async(
        {
            "contents": "이 PDF의 제목을 한글 한 줄로.",
            "model": "gemini-2.5-flash",
            "max_tokens": 100,
        },
        file_paths=[LARGE_PDF],
        database="kw_ai_dev",
    )
    elapsed2 = (time.perf_counter() - start) * 1000

    print_flow("대용량 PDF (2회차 — MDB 캐시 히트)", meta2, elapsed2)
    print(f"  응답: {result2.get('text', '')[:200]}")
    print(f"  속도 비교: {elapsed:.0f}ms → {elapsed2:.0f}ms")

---
## 7. 반환값 구조

모든 `llm_request_async` 함수는 `(result, meta)` 튜플을 반환합니다.

```python
result = {
    "text": "LLM 응답 텍스트",
    "model": "gemini-2.5-flash",
    "usage": {"input_tokens": 10, "output_tokens": 5, "total_tokens": 15},
    "finish_reason": "STOP",
    "request_id": "req-xxx",
    "function_calls": None,
}

meta = {
    "source": "live" | "cache",      # LLM 직접 호출 vs MDB 캐시
    "cache_hit": True | False,
    "cache_key": "sha256 hex (64자)",
    "metadata": {"service_name": "...", "model": "..."},
    "collection": "agent_llm",
}
```

### 캐시 키 동작 원리

`llm_input`에서 15개 필드 전부 추출(미입력/None → `""`) → 값 내부 정규화 → 키 정렬·JSON 직렬화 → SHA-256 해시

- **같은 입력 = 같은 캐시 키** → 캐시 히트
- **파일 첨부 시**: 파일 내용의 checksum이 캐시 키에 포함 → 같은 파일 + 같은 질문 = 캐시 히트 (파일명/URI 무관)

---
# Appendix: 에이전트 개발 패턴

`llm_request_async`를 사용하는 에이전트 개발 시 자주 쓰는 패턴입니다.

## 패턴 A: 단건 처리

단건 LLM 호출 + 캐시. 에이전트 내 단일 LLM 호출의 기본 패턴.

In [ ]:
from llm_request import llm_request_async

start = time.perf_counter()
result_a, meta_a = await llm_request_async(
    {
        "contents": "삼성전자 반도체 사업 현황을 두 줄로 요약해줘.",
        "model": "gemini-2.5-flash",
        "temperature": 0,
        "max_tokens": 150,
        "service_name": "fund_comment",
    },
    database="kw_ai_dev",
)
elapsed = (time.perf_counter() - start) * 1000

print_flow("패턴 A: 단건 처리", meta_a, elapsed)
print(f"  응답: {result_a.get('text', '')[:200]}")
if result_a.get("usage"):
    u = result_a["usage"]
    print(f"  토큰: {u.get('input_tokens', 0)} in / {u.get('output_tokens', 0)} out")

---
## 패턴 B: N건 병렬 — `llm_request_async` + 타이밍 로그

펀드마다 **DB 조회 지연**을 다르게 두어 LLM 요청 시각이 어긋나게 합니다.
로그에 `[경과초] 코드 DB완료` → `LLM시작` → `LLM끝(소요초)` 가 찍혀, **비동기 병렬 진행**을 확인할 수 있습니다.

```
005930: task → DB(즉시) → [LLM 2~4s] ─────────────┐
000660: task → DB(0.4s) → [LLM ...]              ├─ as_completed
035420: task → DB(0.8s) → [LLM ...]              │
068270: task → DB(1.2s) → [LLM ...]              ┘
```

In [ ]:
import asyncio
import time

from llm_request import llm_request_async

FUNDS = [
    {"code": "005930", "name": "삼성전자",  "sector": "반도체"},
    {"code": "000660", "name": "SK하이닉스", "sector": "반도체"},
    {"code": "035420", "name": "NAVER",      "sector": "IT"},
    {"code": "068270", "name": "셀트리온",   "sector": "바이오"},
]

DB_STAGGER_SEC = 0.42


async def process_fund(fund: dict, idx: int, wall0: float) -> dict:
    """DB 조회(시뮬) 후 LLM 호출."""
    # DB 조회 시뮬레이션 (stagger)
    await asyncio.sleep(DB_STAGGER_SEC * idx)
    t_db_done = time.perf_counter() - wall0
    print(f"  [{t_db_done:.2f}s] {fund['code']} DB 완료")

    # LLM 호출
    t_llm_start = time.perf_counter() - wall0
    result, meta = await llm_request_async(
        {
            "contents": (
                f"{fund['name']}({fund['code']})의 {fund['sector']} 사업 현황을 "
                f"투자 관점에서 간결한 한 문단(3~4문장)으로 설명해줘."
            ),
            "model": "gemini-2.5-flash",
            "temperature": 0.3,
            "max_tokens": 300,
            "service_name": "fund_comment",
        },
        database="kw_ai_dev",
    )
    t_llm_end = time.perf_counter() - wall0
    llm_sec = t_llm_end - t_llm_start

    print(f"  [{t_llm_end:.2f}s] {fund['code']} LLM 끝 ({llm_sec:.2f}s, {meta.get('source', '?')})")

    return {
        "code": fund["code"],
        "name": fund["name"],
        "text": result.get("text", ""),
        "source": meta.get("source", "?"),
        "t_db_done": t_db_done,
        "t_llm_start": t_llm_start,
        "t_llm_end": t_llm_end,
        "llm_sec": llm_sec,
    }


# 실행
wall0 = time.perf_counter()
tasks = [asyncio.create_task(process_fund(f, i, wall0)) for i, f in enumerate(FUNDS)]
many_results = []
for coro in asyncio.as_completed(tasks):
    many_results.append(await coro)

total = time.perf_counter() - wall0
print(f"\n  전체 소요: {total:.2f}s (순차 시 ~{sum(r['llm_sec'] for r in many_results):.2f}s)")

### 결과 확인 — 요약 표

`LLM시작` 시각이 DB끝 직후에 연달아 찍히면 비동기 병렬 진행 확인.

In [ ]:
rows = sorted(many_results, key=lambda x: x["code"])
hdr = f"{'코드':<8} {'DB끝(s)':>8} {'LLM시작(s)':>10} {'LLM끝(s)':>9} {'순수LLM(s)':>10} {'source':<8}"
print(hdr)
print("-" * len(hdr))
for r in rows:
    print(
        f"{r['code']:<8} "
        f"{r['t_db_done']:8.2f} "
        f"{r['t_llm_start']:10.2f} "
        f"{r['t_llm_end']:9.2f} "
        f"{r['llm_sec']:10.2f} "
        f"{r['source']:<8}"
    )

print()
for r in rows:
    preview = r["text"].replace("\n", " ")[:100]
    print(f"[{r['code']}] {preview}")

---
## 정리

테스트 데이터를 삭제합니다. (kw_ai_dev DB)

In [ ]:
# MDB 캐시 정리 — done_check으로 cache_key를 구해서 삭제 요청
# (서버화 후에는 직접 MDB 접근이 아닌, 서버 API를 통해 정리)

from llm_request import done_check

test_inputs = [
    {"contents": "1+1은?", "model": "gemini-2.5-flash"},
    {"contents": "ESG 평가 기준을 알려줘", "model": "gemini-2.5-flash", "system": "너는 ESG 전문가야. 한국어로 답변해."},
    {"contents": "삼성전자 반도체 사업 현황을 두 줄로 요약해줘.", "model": "gemini-2.5-flash"},
]

# 펀드 패턴 B 입력도 추가
for fund in FUNDS:
    test_inputs.append({
        "contents": (
            f"{fund['name']}({fund['code']})의 {fund['sector']} 사업 현황을 "
            f"투자 관점에서 간결한 한 문단(3~4문장)으로 설명해줘."
        ),
        "model": "gemini-2.5-flash",
    })

print("캐시 키 조회 중...")
for inp in test_inputs:
    dc = await done_check(inp, database="kw_ai_dev")
    status = "hit" if dc["cache_hit"] else "miss"
    print(f"  {dc['cache_key'][:16]}... ({status})")

print("\n정리 완료 (실제 삭제는 MDB에서 직접 수행)")